# Data exploration: Archetypometrics

Following the methodology of Archetypometrics paper:
- **Matrix A:** 464 traits x 2000 characters.
- **Normalization:** Scores are normalized to range [-0.5, 0.5].
- **SVD:** Performed without centering the data.

## Import

In [93]:
import pandas as pd
import os
import numpy as np
pd.set_option('display.max_colwidth', None) #full display for links

## Loading data

In [110]:
# Define the path to the data
data_dir = 'data/plain/current/2000/'

# Load metadata
stories_df = pd.read_csv(os.path.join(data_dir, 'data_stories.tsv'), sep='\t', header = None)
characters_df = pd.read_csv(os.path.join(data_dir, 'data_characters.tsv'), sep='\t', header = 0)
traits_df = pd.read_csv(os.path.join(data_dir, 'data_traits.tsv'), sep='\t', header = 0)

# Load SVD components
U = np.loadtxt(os.path.join(data_dir, 'data_archetype_space_U.txt'))       # (2000, 464)
raw_sigma = np.loadtxt(os.path.join(data_dir, 'data_archetype_space_Sigma.txt')) # (464,)
V = np.loadtxt(os.path.join(data_dir, 'data_archetype_space_V.txt'))    

#Load raw A matrix
A_raw = np.loadtxt(os.path.join(data_dir, 'data_raw_A.txt'))
A = A_raw.T #transpose data

## Preview data

In [111]:
stories_df.head(5)

,0,1,2,3
0,1,(500) Days of Summer,My Little Pony: Friendship Is Magic,https://compstorylab.org/archetypometrics/cards/My-Little-Pony-Friendship-Is-Magic-2000-464-341.pdf
1,2,10 Things I Hate About You,NCIS,https://compstorylab.org/archetypometrics/cards/NCIS-2000-464-341.pdf
2,3,30 Rock,Wynonna Earp,https://compstorylab.org/archetypometrics/cards/Wynonna-Earp-2000-464-341.pdf
3,4,8 Mile,The Walking Dead,https://compstorylab.org/archetypometrics/cards/The-Walking-Dead-2000-464-341.pdf
4,5,A Nightmare on Elm Street,White Collar,https://compstorylab.org/archetypometrics/cards/White-Collar-2000-464-341.pdf


In [112]:
print(f"Sanity check for number of stories:{stories_df.shape}")

Sanity check for number of stories:(341, 4)


In [113]:
characters_df.head(5)

,index,character,character in latex,character/story,character/story in latex,card url
0,1,Applejack,Applejack,Applejack/My Little Pony: Friendship Is Magic,Applejack/My Little Pony: Friendship Is Magic,https://compstorylab.org/archetypometrics/cards/My-Little-Pony-Friendship-Is-Magic-Applejack-2000-464-341.pdf
1,2,Twilight Sparkle,Twilight Sparkle,Twilight Sparkle/My Little Pony: Friendship Is Magic,Twilight Sparkle/My Little Pony: Friendship Is Magic,https://compstorylab.org/archetypometrics/cards/My-Little-Pony-Friendship-Is-Magic-Twilight-Sparkle-2000-464-341.pdf
2,3,Rarity,Rarity,Rarity/My Little Pony: Friendship Is Magic,Rarity/My Little Pony: Friendship Is Magic,https://compstorylab.org/archetypometrics/cards/My-Little-Pony-Friendship-Is-Magic-Rarity-2000-464-341.pdf
3,4,Pinkie Pie,Pinkie Pie,Pinkie Pie/My Little Pony: Friendship Is Magic,Pinkie Pie/My Little Pony: Friendship Is Magic,https://compstorylab.org/archetypometrics/cards/My-Little-Pony-Friendship-Is-Magic-Pinkie-Pie-2000-464-341.pdf
4,5,Spike,Spike,Spike/My Little Pony: Friendship Is Magic,Spike/My Little Pony: Friendship Is Magic,https://compstorylab.org/archetypometrics/cards/My-Little-Pony-Friendship-Is-Magic-Spike-2000-464-341.pdf


In [98]:
print(f"Sanity check for number of characters:{characters_df.shape}")

Sanity check for number of characters:(2001, 6)


In [99]:
traits_df.head(5)

,index,differential,flipped differential,left pole,right pole,card url
0,1,playful :: serious,serious :: playful,playful,serious,https://compstorylab.org/archetypometrics/cards/playful--serious-2000-464-341.pdf
1,2,shy :: bold,bold :: shy,shy,bold,https://compstorylab.org/archetypometrics/cards/shy--bold-2000-464-341.pdf
2,3,cheery :: sorrowful,sorrowful :: cheery,cheery,sorrowful,https://compstorylab.org/archetypometrics/cards/cheery--sorrowful-2000-464-341.pdf
3,4,masculine :: feminine,feminine :: masculine,masculine,feminine,https://compstorylab.org/archetypometrics/cards/masculine--feminine-2000-464-341.pdf
4,5,charming :: awkward,awkward :: charming,charming,awkward,https://compstorylab.org/archetypometrics/cards/charming--awkward-2000-464-341.pdf


In [100]:
print(f"Sanity check for number of traits:{traits_df.shape}")

Sanity check for number of traits:(464, 6)


## Understanding Matrix A

In [101]:
A_raw

array([[-0.096,  0.201,  0.13 , ...,  0.277,  0.208, -0.004],
       [ 0.38 ,  0.06 ,  0.321, ..., -0.13 ,  0.197, -0.251],
       [-0.331, -0.184, -0.173, ...,  0.105,  0.037,  0.075],
       ...,
       [-0.043,  0.361,  0.471, ...,  0.291,  0.356,  0.117],
       [-0.302,  0.016, -0.001, ..., -0.159, -0.211,  0.278],
       [ 0.009, -0.34 , -0.263, ..., -0.439,  0.019, -0.256]],
      shape=(464, 2000))

In [102]:
print(f"Matrix A_raw shape: {A_raw.shape}")
print(f"Value range: [{np.min(A_raw)}, {np.max(A_raw)}]")

Matrix A_raw shape: (464, 2000)
Value range: [-0.489, 0.498]


+ Each entry represents the average rating for a specific character on a specific trait, normalized to a range of -0.5 to 0.5.
+ Poles: 2 poles ( towards -0.5 aligns with the left pole, towards + 0.5 aligns with right pole) <br>

**Tranpose matrix A** <br>
+ New dimensions: 2000 x 4, where each row represents a single character and each column  represents a specific trait.
+ The coordinates for a character in "trait space" show how their persionality is perceived according to the ratings. Used for SVD
<br>
**So $A$ organizes the data by trait, while $A^T$ organizes the data by character.**

## Understand global variance

+ The "file data_archetype_space_Sigma.txt" contains the singular values
($\sigma_i$). In SVD, the variance explained by the $i$-th archetype (or component) is proportional to the square of its singular value ($\sigma_i^2$).
+ The first singular value ($99.6627$) is much larger than the rest, meaning the first few archetypes capture the vast majority of the personality differences across all characters.

In [103]:
if len (raw_sigma.shape) >1: 
    sigma = np.diag(raw_sigma)
else:
    sigma = raw_sigma

In [104]:
variance_explained = (sigma ** 2) / np.sum(sigma ** 2)
cumulative_variance = np.cumsum(variance_explained)
print(f"  First singular value:  {sigma[0]}")
print(f"  All non-negative:      {np.all(sigma >= 0)}")
print(f"  Sorted descending:     {np.all(np.diff(sigma) <= 0)}")
print("\n=== Variance Explained ===")
for k in [1, 5, 10, 20, 50]:
    print(f"  Top {k:>3} archetypes: {cumulative_variance[k-1]*100:.1f}%")

  First singular value:  99.6627
  All non-negative:      True
  Sorted descending:     True

=== Variance Explained ===
  Top   1 archetypes: 24.4%
  Top   5 archetypes: 70.8%
  Top  10 archetypes: 80.8%
  Top  20 archetypes: 85.6%
  Top  50 archetypes: 89.9%


**Sanity check**
+ Sigular values represent magnitudes/distancs - can not be negative
+ SVD algorithm output singular values from largest to smallest. True - archetype is in order.
+ Big "jump" - Going from 1 to 5 archetypes adds 46% more variance. Going from 10 to 50 archetypes only adds 9% more variance. The first 5–10 archetypes are the real personality while the rest might be the quirks 

## Check archetypes and top traits for them

In [105]:
traits_df.columns

Index(['index', 'differential', 'flipped differential', 'left pole',
       'right pole', 'card url'],
      dtype='object')

In [106]:
trait_names = traits_df['differential'].tolist()
with open(data_dir + "data_archetype_space_archetype_classes.txt", 'r') as f:
    archetypes = [line.strip() for line in f.readlines()]

In [107]:
def get_archetype_spectrum(dim_index, pos_label, neg_label):
    """ Extract the traits that define the two poles of an SVD dimension"""
    column = U[:, dim_index]
    # Sort traits by their "weight" (loading) in this dimension
    sorted_indices = np.argsort(column)
    print(f"DIMENSION {dim_index + 1}: {pos_label} (+) vs. {neg_label}(-)")
    # Top positive traits (high weights)
    print(f"\n[+] {pos_label} POLE (Top 10 Traits):")
    for i in reversed(sorted_indices[-10:]):
        print(f" {column[i]:.4f} | {trait_names[i]}")
    # Top negative traits (low weights)
    print(f"\n[-] {neg_label} POLE (Top 10 Traits):")
    for i in sorted_indices[:10]:
        print(f" {column[i]:.4f} | {trait_names[i]}")  

In [108]:
get_archetype_spectrum(0, "HEROES", "FOOLS")
get_archetype_spectrum(1, "ANGELS", "DEMONS")
get_archetype_spectrum(2, "TRADITIONALISTS", "ADVENTURERS")

DIMENSION 1: HEROES (+) vs. FOOLS(-)

[+] HEROES POLE (Top 10 Traits):
 0.1102 | unambitious :: driven
 0.1003 | disorganized :: self-disciplined
 0.0979 | noob :: pro
 0.0971 | slugabed :: go-getter
 0.0945 | gross :: hygienic
 0.0906 | random :: pointed
 0.0894 | pointless :: meaningful
 0.0851 | dunce :: genius
 0.0842 | racist :: egalitarian
 0.0838 | shy :: bold

[-] FOOLS POLE (Top 10 Traits):
 -0.1181 | diligent :: lazy
 -0.1164 | persistent :: quitter
 -0.1125 | motivated :: unmotivated
 -0.1085 | competent :: incompetent
 -0.1082 | high IQ :: low IQ
 -0.1040 | focused :: absentminded
 -0.1034 | resourceful :: helpless
 -0.1032 | perceptive :: unobservant
 -0.1017 | workaholic :: slacker
 -0.0948 | overachiever :: underachiever
DIMENSION 2: ANGELS (+) vs. DEMONS(-)

[+] ANGELS POLE (Top 10 Traits):
 0.1038 | forgiving :: vengeful
 0.0992 | angelic :: demonic
 0.0972 | empath :: psychopath
 0.0964 | wholesome :: salacious
 0.0920 | complimentary :: insulting
 0.0898 | trusting :

## Moral Vector Projection (3 traits)
+ Decide on a set of 1-3 traits, out of the 464, that we deem to be our "moral" traits
+ Instead of our $A$ matrix ($464 \times 2000$), make a moral ($A_m$, size $464 \times 1$), where all traits are zeroed out except for our "moral" trait set, which are dialed all the way up in magnitude (optional: or maybe not all the way up? Are they ordered?) 
+ $U^T \times A_m$ gives us the "moral" axis in archetype space. 
+Project characters onto it to find moral/immoral characters. How well do these align with our intuition?

### Forgiving - Angelic - Empath

In [121]:
##Code 
##Choosing traits 
moral_3_trait_indices = [
    traits_df[traits_df['differential'] == 'forgiving :: vengeful'].index[0],
    traits_df[traits_df['differential'] == 'angelic :: demonic'].index[0],
    traits_df[traits_df['differential'] == 'empath :: psychopath'].index[0]
]

# Create vector
Am = np.zeros((464, 1))
for idx in moral_trait_indices:
    # good' pole is the left one in this data
    Am[idx] = -0.5

#Find the axis in archetype space
# w_moral = U^T * Am
w_moral = np.dot(U.T, Am)

#Project 
# character_scores = V * w_moral
V_reduced = V[:, :464]
character_coords = V_reduced * sigma[:464]
character_scores = np.dot(character_coords, w_moral).flatten()

# Results
characters_df['moral_score'] = character_scores
top_moral = characters_df.sort_values('moral_score', ascending=False).head(10)
top_immoral = characters_df.sort_values('moral_score', ascending=True).head(10)

print("=== Top moral characters (Forgiving - Angelic - Empath) ===")
print(top_moral[['character', 'moral_score']])

=== Top moral characters (Forgiving - Angelic - Empath) ===
           character  moral_score
1708    Sam Obisanya     0.647749
201         Pop Tate     0.630473
1113       Mamá Coco     0.615554
1362            Aang     0.614047
1368            Iroh     0.609426
913      Deanna Troi     0.608365
160       Beth March     0.605738
440            Penny     0.605692
1140  Alphonse Elric     0.599507
1699       Ted Lasso     0.599363


## Moral Vector Projection (4 Traits)
### Honorable, Loyal, Heroic, and Forgiving

In [122]:
##Choosing traits 
moral_4_trait_indices = [
    traits_df[traits_df['differential'] == 'honorable :: cunning'].index[0],
    traits_df[traits_df['differential'] == 'loyal :: traitorous'].index[0],
    traits_df[traits_df['differential'] == 'heroic :: villainous'].index[0], 
    traits_df[traits_df['differential'] == 'forgiving :: vengeful'].index[0]
]
# Create vector
Am_2 = np.zeros((464, 1))
for idx in moral_4_trait_indices:
    # good' pole is the left one in this data
    Am_2[idx] = -0.5

#Find the axis in archetype space
# w_moral = U^T * Am
w_moral_2 = np.dot(U.T, Am)

#Project 
# character_scores = V * w_moral
V_reduced_2 = V[:, :464]
character_coords_2 = V_reduced_2 * sigma[:464]
character_scores_2 = np.dot(character_coords, w_moral).flatten()

# Results
characters_df['moral_score'] = character_scores_2
top_moral_2 = characters_df.sort_values('moral_score', ascending=False).head(10)
top_immoral_2 = characters_df.sort_values('moral_score', ascending=True).head(10)

print("=== Top moral characters (Forgiving - Angelic - Empath) ===")
print(top_moral_2[['character', 'moral_score']])

IndexError: index 0 is out of bounds for axis 0 with size 0

## Flexible Trait Exploration (Up to 6 Traits)
Pick your traits and directions (-1 for Left Pole, +1 for Right Pole).

In [ ]:
# CUSTOMIZE YOUR AXIS HERE
custom_traits = {
    38: -1, # heroic :: villainous (Target: Heroic)
    83: +1, # cruel :: kind (Target: Kind)
    78: +1, # selfish :: altruistic (Target: Altruistic)
    80: -1, # angelic :: demonic (Target: Angelic)
    13: -1, # honorable :: cunning (Target: Honorable)
    63: +1  # debased :: pure (Target: Pure)
}

w_custom = np.zeros(464)
print("Traits selected:")
for idx, direction in custom_traits.items():
    row = traits_df.iloc[idx]
    target = row['left pole'] if direction == -1 else row['right pole']
    print(f"- {row['differential']} (Point towards: {target})")
    w_custom[idx] = direction

w_custom /= np.linalg.norm(w_custom)
characters_df['custom_axis_score'] = A_ct @ w_custom

print("\n--- Top 10 Characters Aligned with Custom Axis ---")
display(characters_df.sort_values('custom_axis_score', ascending=False).head(10)[['character', 'character/story', 'custom_axis_score']])

## SVD Analysis

In [ ]:
U, S, Vt = np.linalg.svd(A, full_matrices=False)
print(f"Top 5 Singular Values: {S[:5]}")

# Check alignment with PC1
alignment = np.dot(Vt[0], w_custom)
print(f"Alignment of Custom Axis with PC1: {alignment:.4f}")